In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 4
:caption: Contents:

# 2D case: walkthrough example


This notebook walks through a complete 2D XBeach case preparation using the `oceanicospy.models.xbeachpy` subpackage. The example is based on a **non-stationary** hydrodynamic simulation over a 2D coastal domain (Sound Bay, Caribbean coast of Colombia, May 2025).

## Workflow overview

This example user certain input files which has to be placed in `<path_case>/input/`:

| File | Description |
|---|---|
| `XBeach_domain.shp` | Polygon shapefile defining the 2D model extent |
| `TopoBathy*.csv` | X, Y, Z topobathymetry on a regular grid |
| `SpecSWAN.out` | SWAN spectral output at boundary points |
| `points_output.txt` | (optional) list of gauge output X/Y coordinates |

ERA5 winds and UHSLC water levels are **downloaded automatically** if not already cached.

## Imports

In [1]:
from oceanicospy.models import xbeachpy

from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Case configuration

Three main parameters are required to set up a XBeach case:

| Parameter | Description |
|---|---|
| `path_case` | Root directory that will hold `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` / `end_date` | Simulation window |

All user-facing settings live in plain dictionaries. The user-defined dictionary `dict_ini_data` is passed to the `Initializer` class which creates the folder structure and stamps the `params.txt` file with these settings. The expected keys for this dictionary are:

- ``case_description``: Free-form text description of the case (not used in XBeach)
- ``act_morf``: Morphological activation (0 = off)
- ``act_sedtrans``: Sediment transport activation (0 = off)
- ``act_wavemodel``: Spectral wave model activation (1 = surfbeat)
- ``dims``: Number of dimensions (2 = 2D)

In [2]:
# ── path to the case root ─────────────────────────────────────────────────────
path_case = '../../data/model_runs/2D_xbeach/' # It has to end with a slash. The folder structure will be created inside this path.

# ── case-level flags written into params.txt ──────────────────────────────────
ini_case_data = dict(
    case_description='2D Sound Bay - San Andres',
    act_morf=0,
    act_sedtrans=0,
    act_wavemodel=1,
    dims=2
)

# ── simulation period ─────────────────────────────────────────────────────────
ini_date = datetime(2025, 5, 9, 4)
end_date = datetime(2025, 5, 10, 4)

## Initialization

`Initializer` does two things:

1. **`create_folders`**: creates the project directory tree: `input/`, `pros/`, `run/`, and `output/` under `path_case`.  
   If `run/` or `output/` already exist they are **wiped and re-created** to avoid stale files from a previous attempt.

2. **`replace_ini_data`**: copies the bundled `params_base.txt` template into `run/params.txt` and substitutes the flags from `ini_case_data`.  
   Any key not supplied by the user falls back to the package defaults in `xbeachpy/utils/defaults.py`.

In [5]:
case = xbeachpy.Initializer(
    root_path=path_case,
    dict_ini_data=ini_case_data,
    ini_date=ini_date,
    end_date=end_date
)
case.create_folders()
case.replace_ini_data()

*** Initializing XBeach model ***


	*** Creating project folder structure ***


	*** Copying base XBeach configuration file into run folder ***



After this step the folder tree looks like:

```
path_case/
├── input/           ← place your static input files here
├── pros/
├── run/
│   └── params.txt   ← generated from the bundled template
└── output/
```

## Generating the grid

`xbeachpy` has a flexible grid generation utility called `GridMaker` that can build uniform or segmented grids in 1D or 2D. For this example we will build a uniform 2D grid based on the bounding box of a shapefile. 

For this purpose method from `GridMaker.build_rectangular_grid()` which reads the bounding box of a user-defined shapefile placed in the input folder and lays out a uniform 2D grid. The output files from the grid generator methods are `x.grd` and `y.grd`; they are written into `run/`.

```{hint}
if you already have pre-built `.grd` files for x and y direction, you can place both inside `input/` and `GridMaker` will detect them, copy them to `run/`, and skip grid construction entirely.
```

In [15]:
case_grid = xbeachpy.preprocess.GridMaker(init=case,dx=10, dy=10)

In [16]:
case_grid.build_rectangular_grid(source_file='XBeach_domain.shp')

In [17]:
case_grid.metadata

{'xfilepath': 'x.grd', 'yfilepath': 'y.grd', 'meshes_x': 100, 'meshes_y': 110}

In [ ]:
# case_grid = xbeachpy.preprocess.GridMaker(init=case, dx=10, dy=10)
# case_grid.rectangular(source_file='XBeach_domain.shp')
# case_grid.fill_grid_section()


*** Adding/Editing grid information in params file ***



If you want to check the metadata was entered into the `params.txt` file you can use a property called `metadata`

In [6]:
print(case_grid.metadata)

{'xfilepath': 'x.grd', 'yfilepath': 'y.grd', 'meshes_x': 100, 'meshes_y': 110}


The `metadata` dictionary contains the relevant information about the grid that was either generated or copied from `input/`. The expected keys are:

```code_block:: python
{
    'xfilepath': 'x.grd',
    'yfilepath': 'y.grd',
    'meshes_x' : <int>,   # number of cells in the x-direction
    'meshes_y' : <int>    # number of cells in the y-direction
}
```

## Setting up the bathymetry

The `BathyMaker` class provides methods for preprocessing bathymetry data. The `BathyMaker.convert_xyz2asc()` method reads a **regular-grid** topobathymetry CSV (named `TopoBathy*.csv`) from `input/`, pivots it into a 2D array ordered in XBeach convention, and saves it as an ASCII `.dep` file in `run/`.

**Expected CSV format:**

```
x_coordinate,y_coordinate,z
762000,1385000,-12.5
762010,1385000,-12.3
...
```
Positive z coordinate means depths in meters. NaN cells (gaps in the grid) are written as empty fields in the `.dep` file.

In [7]:
case_bathy = xbeachpy.preprocess.BathyMaker(init=case, output_filename='bathymetry')
case_bathy.convert_xyz2asc(xyz_filename='TopoBathy_XBeach_domain_Lidar&10m&50m')
case_bathy.fill_bathy_section()

File ../../data/model_runs/2D_xbeach/run/bathymetry.dep saved successfuly.

*** Adding/Editing bathymetry information in params file ***



## Configurating wind forcing (ERA5)

`WindForcing` class integrates directly with different data fetcher classes like `oceanicospy.downloads.ERA5Downloader`:

Similar to `swanpy` subpackage, `WindForcing` has two main methods for handling ERA5 wind data:
- **`get_winds_from_ERA5()`** — downloads hourly U10/V10 from the CDS API for the bounding box defined in `wind_dict`. If the NetCDF file already exists in `input/` the download is skipped.
- **`write_ERA5_ascii()`** — converts the NetCDF to a two-column (time [s], speed [m/s], direction [°N]) ASCII file and copies/links it into `run/`.  

The `wind_dict` should contain the following keys similar to the one used in `swanpy`:

- `lon_ll_corner_wind`: longitude of the lower-left corner of the wind grid
- `lat_ll_corner_wind`: latitude of the lower-left corner of the wind grid
- `nx_wind`: number of grid points in the x-direction (longitude)
- `ny_wind`: number of grid points in the y-direction (latitude)
- `dx_wind`: grid spacing in the x-direction (degrees)
- `dy_wind`: grid spacing in the y-direction (degrees)

In [6]:
wind_dict = dict(lon_ll_corner_wind=278, 
                 lat_ll_corner_wind=12.3, 
                 nx_wind=24, 
                 ny_wind=24, 
                 dx_wind=0.025,
                 dy_wind=0.025)

In [7]:
case_winds = xbeachpy.preprocess.WindForcing(
    init=case,
    wind_info=wind_dict,
    use_link=False
)


*** Initializing winds ***



```{hint}
`use_link=False` copies the file; `use_link=True` creates a symlink (saves space on filesystems).

Once the class is instantiated, the two methods mentioned above are called sequentially to handle the download and processing of the wind data.

In [9]:
case_winds.get_winds_from_ERA5(utc_offset_hours=-5,format_localtime=True)

2026-04-28 04:03:06,144 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

a4112ebde1b05589338a1a4643713a9a.nc:   0%|          | 0.00/197k [00:00<?, ?B/s]

Downloaded winds_era5.nc to ../../data/model_runs/2D_xbeach/input/winds_era5.nc.
	 ERA5 wind data downloaded successfully


ERA5 and CMDS downloader classes are designed to format the time in local time if `format_localtime=True` is passed as an argument. This means that the downloaded files will have their timestamps converted to local time based on the provided `utc_offset_hours`. If users do not choose to format the time in local time, they should be aware that the timestamps in the downloaded files will be in UTC, which may require additional handling when the `write_ERA5_ascii()` be called to convert the NetCDF file to XBeach ASCII format, as the time formatting in the output file will depend on the original timestamps in the NetCDF file.
```

```{info}
XBeach does not use wind fields in the same way as SWAN. Instead of spatially distributed wind forcing, XBeach expects a single time series of wind speed and direction at a representative location. Therefore, the `write_ERA5_ascii()` method extracts the wind data at a single grid point (either specified by the user or defaulting to the last point in the dataset) and writes it in a format suitable for XBeach.

In [10]:
case_winds.write_ERA5_ascii(
    era5_filename='winds_era5_localtime.nc',
    ascii_filename='winds.wnd',
    lon_target=-81.706,
    lat_target=12.5204
)
case_winds.fill_wind_section()

	 ERA5 wind data converted to ASCII format and saved as winds.wnd

*** Adding/Editing winds information in params file ***



##  Setting up the water level forcing

`WaterLevelForcing` is a utility that connects to the **University of Hawaii Sea Level Center (UHSLC)** research-quality gauge archive and downloads water level data for a specified station and time window.



The following steps are involved in setting up the water level forcing:

1. **Downloading** hourly tide-gauge data for the simulation window from UHSLC.
2. **Converting** the CSV to a time series XBeachASCII water-level file (one value per timestep).
3. **Writing** the tide section of `params.txt` to point to the generated water level file.

The San Andres UHSLC station code is **737** according to the [UHSLC station list](https://uhslc.soest.hawaii.edu/stations/).

In [12]:
case_wl = xbeachpy.preprocess.WaterLevelForcing(init=case, use_link=False)
df_wl = case_wl.get_waterlevel_from_UHSLC(station_id=737)


*** Initializing water levels ***

the file ../../data/model_runs/2D_xbeach/input/h737.csv already exists
	 UHSLC water level data already exists, skipping download


```{warning}
Unlike the `WindForcing` class, the method `get_waterlevel_from_UHSLC` in the `WaterLevelForcing` class returns a data structure, not a file. This is because sometimes the pd.DataFrame containing the water level data needs to be manipulated before being written to an ASCII file and this manipulation can be case-specific per station.

Particularly for this station, it was found that the water level data from station `737` was affected by a sudden -2 m shift between 1997 and 2018. This was likely due to a change in the reference point of the gauge or a similar issue. To correct for this, a mask was applied to the DataFrame to identify the affected time range and subtract 2 m from the depth values during that period.
```

In [13]:
# mask application
correction_mask = (
    (df_wl.index >= datetime(1997, 1, 1, 0)) &
    (df_wl.index <= datetime(2018, 12, 31, 18))
)
df_wl.loc[correction_mask, "depth[m]"] -= 2.0

case_wl.write_UHSLC_ascii(df_wl, 'water_levels.wl')
case_wl.fill_wl_section()

	 UHSLC water level data converted to ASCII format and saved as water_levels.wl

*** Adding/Editing water level information in params file ***



## Defining the wave boundary conditions

This case uses non-stationary wave boundary conditions derived from a SWAN spectral output file (`SpecSWAN.out`). The `BoundaryConditions` class provides a method called `spectra_from_swan()` that processes this file and prepares the necessary input for XBeach.


1. In ordet to follows the XBeach conventions, each site must has its time time series as individual `.sp2` files. Additionally, each boundary point will have its own directory `run/bounds_conds/point_<i>/` where `<i>` is the point number. All the `.sp2` files must be placed inside the corresponding `point_<i>/` directory.
2. All the .sp2 files are written to their respective directories and they are properly referenced in the `filelist_<i>.txt` files.
3. Finally, a `loclist.txt` maps each boundary location to its corresponding filelist.

In [14]:
case_bounds = xbeachpy.preprocess.BoundaryConditions(init=case)

*** Cleaning Boundary Conditions ***
*** Initializing Boundary Conditions ***


We invoke the `spectra_from_swan()` method to process the SWAN output and generate the necessary files for XBeach. Assuming the SWAN output contains spectral data for multiple boundary points, the method will iterate through each point, extract the relevant time series, and save them in the appropriate format and directory structure required by XBeach. The input filename where the swan outputs are stored is passed as an argument to the method including its extension (e.g., `SpecSWAN.out`). This input file is expected to be placed in the `input/` directory of the case.

In [15]:
case_bounds.spectra_from_swan(input_filename='SpecSWAN.out',point_indexes=range(3,15)) 

We can also use the `point_indexes` argument to specify which points from the SWAN output should be processed. If `point_indexes` is not provided, the method will process all available points in the SWAN output. As the name says, the indexes in `point_indexes` should correspond to the position of the points in the SWAN output file, starting from 0. For example, if you want to process the first three points, you would use `point_indexes=range(3)`. If you want to process points 3 to 14, you would use `point_indexes=range(3,15)`.

We can finally replace the required placeholders in the `params.txt` file with the corresponding flags for the wave boundary conditions.

In [16]:
case_bounds.fill_boundaries_section()


*** Adding/Editing boundary information for domain in configuration file ***



## Output and compilation configuration

For this section the parameters for the compilation and output settings for the case have to be defined.

`CaseRunner` finalises `params.txt` with the required output and computation settings:

| Method | Purpose |
|---|---|
| `write_output_file()` | set the output NetCDF filename |
| `write_output_points()` | read gauge coordinates from `points_output.txt` and embed them |
| `select_global_vars()` | choose 2D/3D field variables written at `tintg` |
| `select_point_vars()` | choose time-series variables at the gauge locations |
| `fill_slurm_file()` | copy the SLURM template and stamp paths + case name |
| `fill_computation_section()` | compute `tstop` from the start/end dates and write remaining params |

> `fill_computation_section()` must be called **last** — it converts the datetime strings to a `tstop` value (seconds) and finalises the file.

Before the compilation step, a compilation data dictionarty must be defined with the following keys:

| Key | Meaning |
|---|---|
| `tint_value` | Output interval for time-series data at points (seconds) |
| `tintg_value` | Output interval for global 2D/3D fields (seconds) |

In [17]:
# ── computation / output settings ─────────────────────────────────────────────
comp_data_nonstat = dict(
    tint_value=3600,    # output interval [s]
    tintg_value=3600,   # global output interval [s]
)

Once the `comp_data_nonstat` dictionary is defined, the `CaseRunner` methods are called sequentially to set up the output configuration and finalise the `params.txt` file for compilation.

In [18]:
case_output = xbeachpy.execution.CaseRunner(
    init=case,
    dict_comp_data=comp_data_nonstat
)


*** Initializing Case Runner ***



If the user wants to get output at specific points, a file named `points_output.txt` must be placed in the `input/` folder with the following format:

```x_coordinate,y_coordinate
762500,1385500
762600,1385500
...
```

The coordinates must follow the model grid reference system.


In [19]:
# filepath for the output file with the global 2D/3D fields (netCDF format)
case_output.write_output_file(filename='SoundBay2D.nc')

# selection of the variables to be included in the global output file
case_output.select_global_vars(list_vars=['zs','hh','zb0','H','u','v'])

# definition of output points
case_output.write_output_points(filename='points_output.txt')

# selection of the variables to be included in the output file with time-series at points
case_output.select_point_vars(list_vars=['zs','hh','zb0','H','u','v'])

```{note}
The variable names to be included in the output files can be selected from the list of available variables in XBeach. That is, they follow the same naming convention.

In [20]:
case_output.fill_computation_section()

## Post-processing (after the run)

Once XBeach has finished, `OutputReader` lazily opens the NetCDF output and splits it into:
- **field output** — variables with more than 2 dimensions (time × y × x)
- **point output** — variables with exactly 2 dimensions (time × n_points)

In [21]:
# output_path = f"{case.dict_folders['output']}SoundBay2D.nc"

# reader = xbeachpy.postprocess.OutputReader(filepath=output_path)

# # 2D/3D field variables  (time x y x)
# field_ds = reader.read_field_output()

# # Point time-series variables  (time x n_points)
# point_ds = reader.read_point_output()

In [23]:
# import matplotlib.pyplot as plt

# # Significant wave height at the last output time step
# H_final = field_ds['H'].isel(globaltime=-1)

# fig, ax = plt.subplots(figsize=(9, 6))
# H_final.plot(ax=ax, cmap='viridis', robust=True)
# ax.set_title('Significant wave height — final time step')
# ax.set_xlabel('x [m]')
# ax.set_ylabel('y [m]')
# plt.tight_layout()
# plt.show()

In [24]:
# # Water surface elevation time series at gauge 0
# zs_p0 = point_ds['zs'].isel(points=0)

# fig, ax = plt.subplots(figsize=(11, 3))
# zs_p0.plot(ax=ax, color='steelblue')
# ax.set_title('Water surface elevation — gauge 0')
# ax.set_xlabel('time')
# ax.set_ylabel('zs [m]')
# plt.tight_layout()
# plt.show()


---

### Known limitations

| # | Limitation | Location | Suggested fix |
|---|---|---|---|
| 3 | **`xyz2asc()` assumes a regular grid** — the `pivot_table` step produces NaN-filled rows/columns for scattered or irregular XYZ clouds, leading to a malformed `.dep` file. | `BathyMaker.xyz2asc` | Add a scattered-to-regular interpolation step (e.g. `scipy.interpolate.griddata`) as a fallback for non-regular inputs. |
| 4 | **Wind direction formula flagged as unverified** in the source code (`# this needs to be verified somehow`). The nautical convention `(270 − θ_cart) % 360` may need adjustment depending on the ERA5 u/v axis orientation at a given site. | `WindForcing._ERA5_nc_to_ascii` | Validate against a known NOAA/ECMWF record and add a unit test. |
| 5 | **`OutputReader` is minimal** — variables are separated only by array rank (ndim > 2 vs. ndim == 2). There are no time-slicing, spatial-subsetting, or variable-lookup helpers. | `postprocess.OutputReader` | Add convenience methods such as `get_var(name)`, `sel_time(t0, t1)`, and thin plotting wrappers. |
| 6 | **No CRS management** — the grid, bathymetry, and wind domain are assumed to share the same coordinate reference system. A geographic/projected mismatch produces silently incorrect output. | `GridMaker`, `BathyMaker` | Accept an optional `crs` argument and use `pyproj` to reproject inputs before grid assembly. |